# Experiment: smart use of the dense retrieval budget

**Question.** Dense retrieval (MiniLM-L6) only gets ~256 tokens per doc. Is recall capped by the
*window* or by *how we fill it*? Retrieval wants TOPICALITY (does the doc name the patient's
condition?), not eligibility — and BM25 already indexes the whole doc in the hybrid — so the dense
budget should be topicality-dense, not a truncated full blob.

Three arms, isolating window vs packing:
- **minilm-natural** — current retriever, all fields in natural order, 256 tok (baseline).
- **minilm-prioritized** — same encoder, condition/title/summary-forward, drop `detailed_description`
  + eligibility, 256 tok. Tests *packing* at a fixed window.
- **longctx-prioritized** — a longer-context off-the-shelf encoder at 512 tok, prioritized fields.
  Tests *lifting the window*.

Metric: **recall@100 / recall@1000** on TREC21 (retrieval's job = pool membership), plus the pool's
**oracle NDCG@10** ceiling. NOT reranking NDCG.


## Setup (Colab)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets transformers accelerate pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval, retrieval_blob,
                                 recall_at_k, ndcg_at_k, log_result)
print('loaded')


In [ ]:
# Base + the three arms. Each arm is just an ExperimentConfig override — same harness.
base = ExperimentConfig(data_root=DATA_ROOT)
NATURAL = base.doc_fields
PRIORITIZED = ('conditions', 'brief_title', 'official_title', 'brief_summary', 'interventions')  # topicality; tune me

ARMS = {
  'minilm-natural':     base.with_(retriever_ckpt='semaj83/ctmatch-retriever-v2',
                                   retrieval_fields=NATURAL,     retriever_max_tokens=256),
  'minilm-prioritized': base.with_(retriever_ckpt='semaj83/ctmatch-retriever-v2',
                                   retrieval_fields=PRIORITIZED, retriever_max_tokens=256),
  'longctx-prioritized':base.with_(retriever_ckpt='thenlper/gte-large',   # or BAAI/bge-m3, intfloat/e5-large-v2
                                   retrieval_fields=PRIORITIZED, retriever_max_tokens=512),
}
print({k: c.retriever_ckpt + ' @' + str(c.retriever_max_tokens) for k, c in ARMS.items()})


In [ ]:
# Corpus fields (source of every repr) + TREC21 eval. Full corpus is needed for real recall@1000.
corpus_ids, corpus_fields = load_corpus(base)
sets = load_eval(base, ['trec21'])
rel = sets['trec21']['rel_dict']; topic2text = sets['trec21']['topic2text']
tids = [t for t in rel if t in topic2text]
print(f'{len(corpus_ids):,} docs | {len(tids)} topics')


In [ ]:
# Encode-or-load (cache per arm) + dense retrieve top-1000 over the FULL corpus.
def embed_corpus(name, cfg):
    path = base.path(f'cache/emb_retrieval_{name}.npy')
    if os.path.exists(path):
        return np.load(path).astype(np.float32)
    model = SentenceTransformer(cfg.retriever_ckpt); model.max_seq_length = cfg.retriever_max_tokens
    blobs = [retrieval_blob(f, cfg) for f in corpus_fields]
    emb = model.encode(blobs, convert_to_numpy=True, normalize_embeddings=True,
                       batch_size=64, show_progress_bar=True).astype(np.float32)
    os.makedirs(os.path.dirname(path), exist_ok=True); np.save(path, emb)
    return emb

def retrieve(name, cfg, doc_emb, k=1000):
    model = SentenceTransformer(cfg.retriever_ckpt); model.max_seq_length = cfg.retriever_max_tokens
    q = model.encode([topic2text[t] for t in tids], convert_to_numpy=True,
                     normalize_embeddings=True, batch_size=64).astype(np.float32)
    runs = {}
    for t, qv in zip(tids, q):
        sims = doc_emb @ qv
        top = np.argpartition(-sims, k)[:k]; top = top[np.argsort(-sims[top])]
        runs[t] = [corpus_ids[i] for i in top]
    return runs


In [ ]:
# Run every arm; recall (eligible-only, rel>=2) + oracle NDCG@10 on the retrieved pool.
rows = []
for name, cfg in ARMS.items():
    doc_emb = embed_corpus(name, cfg)
    runs = retrieve(name, cfg, doc_emb)
    r100 = np.mean([recall_at_k(runs[t], rel[t], 100,  rel_level=2) for t in tids])
    r1000 = np.mean([recall_at_k(runs[t], rel[t], 1000, rel_level=2) for t in tids])
    oracle = np.mean([ndcg_at_k(sorted(runs[t], key=lambda d: rel[t].get(d, 0), reverse=True), rel[t]) for t in tids])
    m = {'recall@100': round(float(r100), 3), 'recall@1000': round(float(r1000), 3), 'oracle_ndcg@10': round(float(oracle), 3)}
    log_result(cfg, experiment='retrieval_repr', split='trec21', metrics=m, extra={'arm': name})
    rows.append({'arm': name, **m})
pd.DataFrame(rows)


## Reading it

- **minilm-prioritized > minilm-natural** ⇒ the problem is *packing*: pack topicality, keep MiniLM (cheap).
- **longctx-prioritized > minilm-prioritized** ⇒ the *window* was binding: a bigger encoder is worth it.
- Recall@1000 sets the reranker's oracle ceiling — that's the number to move; oracle_ndcg@10 shows the
  headroom each pool permits. Freeze the winning `retrieval_fields` + `retriever_ckpt` + `retriever_max_tokens`
  into `ExperimentConfig`. (Query-side diagnosis expansion is the separate lever for the implicit-diagnosis
  misses, §8a — not tested here.)
